# 02 — Encoding Vektor SPECTER2 di Google Colab GPU (Hari 2 / Tahap 3)

**Aturan Mutlak Desain Ablasi & Adapter:**
1. Vektor kueri (`query_emb.npy`) **hanya dihitung satu kali** memakai adapter `allenai/specter2_adhoc_query`.
2. Ketiga konfigurasi dokumen (`doc_emb_A/B/C.npy`) **wajib di-encode** memakai adapter `allenai/specter2` (`proximity`) dengan urutan `doc_ids.json` yang identik.

**Sebelum klik `Run All` di Google Colab:**
- Pastikan menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU** sudah aktif.
- Siapkan file `colab_input_bundle.zip` dari folder lokal `metopen-specter2/` (akan diminta unggah otomatis pada Sel 2 bila belum ada di `/content/`).

In [1]:
# 1. Instalasi versi terkunci & pemeriksaan GPU Colab
!pip install -q "adapters==1.3.0" "transformers==4.57.6"

import json
import platform
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import adapters
import numpy as np
import torch
import transformers
from adapters import AutoAdapterModel
from tqdm.auto import tqdm
from transformers import AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU-Fallback"

print("=== Spesifikasi Lingkungan Eksekusi (Catat untuk Section III & README) ===")
print(f"Device Aktif  : {DEVICE} ({GPU_NAME})")
print(f"Python        : {platform.python_version()}")
print(f"PyTorch       : {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"Transformers  : {transformers.__version__}")
print(f"Adapters      : {adapters.__version__}")

if DEVICE == "cuda":
    smi = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    print("nvidia-smi    :", smi.stdout.strip())
else:
    print("PERINGATAN: GPU belum aktif! Aktifkan Runtime -> Change runtime type -> T4 GPU.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
=== Spesifikasi Lingkungan Eksekusi (Catat untuk Section III & README) ===
Device Aktif  : cuda (Tesla T4)
Python        : 3.13.16
PyTorch       : 2.11.0+cu130 (CUDA: 13.0)
Transformers  : 4.57.6
Adapters      : 1.3.0
nvidia-smi    : GPU 0: Tesla T4 (UUID: GPU-9c434cab-df07-d065-e216-f012eb7a2768)


In [2]:
# 2. Unggah & ekstrak colab_input_bundle.zip (atau gunakan folder lokal bila dijalankan di laptop)
IN_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE_BACKUP = False  # Ubah ke True bila ingin menyalin otomatis hasil ke Google Drive

if IN_COLAB:
    WORK_DIR = Path("/content")
    bundle_path = WORK_DIR / "colab_input_bundle.zip"
    if not (WORK_DIR / "data" / "scifact" / "corpus.jsonl").is_file():
        if not bundle_path.is_file():
            from google.colab import files
            print("Silakan pilih dan unggah file 'colab_input_bundle.zip' (~2.67 MB)...")
            uploaded = files.upload()
            for fname in uploaded:
                if fname.endswith(".zip"):
                    bundle_path = WORK_DIR / fname
                    break
        with zipfile.ZipFile(bundle_path, "r") as zf:
            zf.extractall(WORK_DIR)
        print("Ekstraksi colab_input_bundle.zip selesai!")
else:
    WORK_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

from src.data_loader import load_scifact_dataset, validate_scifact_integrity
from src.text_configs import build_all_configs

DATA_DIR = WORK_DIR / "data" / "scifact"
EMB_DIR = WORK_DIR / "embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)

corpus, test_queries, qrels = load_scifact_dataset(DATA_DIR, validate=True)
print("Integritas Dataset Tervalidasi:", validate_scifact_integrity(corpus, test_queries, qrels))

Silakan pilih dan unggah file 'colab_input_bundle.zip' (~2.67 MB)...


Saving colab_input_bundle.zip to colab_input_bundle.zip
Ekstraksi colab_input_bundle.zip selesai!
Integritas Dataset Tervalidasi: {'num_docs': 5183, 'num_test_queries': 300, 'num_relevant_pairs': 339, 'missing_corpus_ids': 0}


In [3]:
# 3. Fungsi utilitas pemuat adapter aman & batch encoding [CLS]
BASE_MODEL = "allenai/specter2_base"
QUERY_ADAPTER = "allenai/specter2_adhoc_query"
DOC_ADAPTER = "allenai/specter2"
MAX_LENGTH = 512
BATCH_SIZE = 32

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
sep_token = tokenizer.sep_token
model = AutoAdapterModel.from_pretrained(BASE_MODEL)

def activate_only_adapter(adapter_model, repo_id: str, adapter_alias: str) -> None:
    """Memuat adapter dari Hugging Face Hub, mengaktifkannya secara eksklusif, dan memindahkan ke GPU."""
    if adapter_alias not in adapter_model.adapters_config.adapters:
        try:
            adapter_model.load_adapter(repo_id, source="hf", load_as=adapter_alias, set_active=True)
        except TypeError:
            adapter_model.load_adapter(repo_id, load_as=adapter_alias, set_active=True)
    adapter_model.set_active_adapters(adapter_alias)
    adapter_model.to(DEVICE)
    adapter_model.eval()

def compute_token_summary(texts: list[str]) -> dict:
    """Menghitung rata-rata token dan persentase > 512 token sebelum pemotongan."""
    enc = tokenizer(texts, add_special_tokens=True, truncation=False, padding=False)
    lengths = np.array([len(ids) for ids in enc["input_ids"]], dtype=int)
    over_limit = int(np.sum(lengths > MAX_LENGTH))
    return {
        "avg_tokens": round(float(lengths.mean()), 2),
        "median_tokens": round(float(np.median(lengths)), 1),
        "max_tokens": int(lengths.max()),
        "truncated_count": over_limit,
        "pct_truncated": round((over_limit / len(lengths)) * 100.0, 2),
    }

def encode_texts_cls(
    texts: list[str],
    batch_size: int = BATCH_SIZE,
    desc: str = "Encoding",
) -> tuple[np.ndarray, float]:
    """Menjalankan batch encoding dengan pengukuran waktu tersinkronisasi CUDA."""
    batches_out = []
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t_start = time.perf_counter()

    with torch.no_grad():
        for start_idx in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch_texts = texts[start_idx : start_idx + batch_size]
            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            outputs = model(**inputs)
            cls_vecs = outputs.last_hidden_state[:, 0, :].detach().cpu().to(torch.float32).numpy()
            batches_out.append(cls_vecs)

    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elapsed_sec = time.perf_counter() - t_start

    full_matrix = np.vstack(batches_out).astype(np.float32)
    return full_matrix, elapsed_sec

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [4]:
# 4. Encoding 300 Kueri Uji (1x saja menggunakan adapter allenai/specter2_adhoc_query)
activate_only_adapter(model, QUERY_ADAPTER, "adhoc_query")
assert "adhoc_query" in str(model.active_adapters), f"Adapter salah: {model.active_adapters}"
print("Adapter kueri aktif:", model.active_adapters)

query_ids = list(test_queries.keys())
query_texts = [test_queries[qid]["text"].strip() for qid in query_ids]
query_tok_stats = compute_token_summary(query_texts)

# Warm-up ringan kueri
_ = encode_texts_cls(query_texts[: min(BATCH_SIZE, len(query_texts))], batch_size=BATCH_SIZE, desc="Warm-up Query")

query_emb, q_time_sec = encode_texts_cls(query_texts, batch_size=BATCH_SIZE, desc="Encode 300 Queries (adhoc_query)")

np.save(EMB_DIR / "query_emb.npy", query_emb)
with (EMB_DIR / "query_ids.json").open("w", encoding="utf-8") as f:
    json.dump(query_ids, f, indent=2)

print(f"Selesai! query_emb.shape = {query_emb.shape} | waktu = {q_time_sec:.3f} s ({(q_time_sec / len(query_ids)) * 1000:.2f} ms/kueri)")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

pytorch_adapter.bin:   0%|          | 0.00/3.59M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Adapter kueri aktif: Stack[adhoc_query]


Warm-up Query:   0%|          | 0/1 [00:00<?, ?it/s]

Encode 300 Queries (adhoc_query):   0%|          | 0/10 [00:00<?, ?it/s]

Selesai! query_emb.shape = (300, 768) | waktu = 0.832 s (2.77 ms/kueri)


In [5]:
# 5. Encoding 5.183 Dokumen Konfigurasi A, B, dan C (menggunakan adapter allenai/specter2 proximity)
activate_only_adapter(model, DOC_ADAPTER, "proximity")
assert "proximity" in str(model.active_adapters), f"Adapter proximity belum aktif: {model.active_adapters}"
assert "adhoc_query" not in str(model.active_adapters), f"Adapter adhoc_query masih aktif: {model.active_adapters}"
print("Adapter dokumen aktif:", model.active_adapters)

doc_ids, config_texts = build_all_configs(corpus, sep_token=sep_token)
assert len(doc_ids) == 5183

# Hitung statistik token di luar blok pengukuran waktu GPU
doc_token_stats = {cfg: compute_token_summary(config_texts[cfg]) for cfg in ["A", "B", "C"]}

# 1x Warm-up pass (32 dokumen) agar inisialisasi kernel CUDA tidak mengotori waktu Konfigurasi A
_ = encode_texts_cls(config_texts["A"][:BATCH_SIZE], batch_size=BATCH_SIZE, desc="CUDA Warm-up Pass (32 docs)")

doc_embeddings = {}
config_logs = {}

for cfg in ["A", "B", "C"]:
    emb_mat, elapsed_sec = encode_texts_cls(
        config_texts[cfg],
        batch_size=BATCH_SIZE,
        desc=f"Encode 5,183 Docs — Config {cfg} (proximity)",
    )
    doc_embeddings[cfg] = emb_mat
    np.save(EMB_DIR / f"doc_emb_{cfg}.npy", emb_mat)

    ms_per_doc = (elapsed_sec / len(doc_ids)) * 1000.0
    config_logs[cfg] = {
        "num_docs": len(doc_ids),
        "total_time_sec": round(float(elapsed_sec), 4),
        "ms_per_doc": round(float(ms_per_doc), 4),
        **doc_token_stats[cfg],
    }
    print(
        f"Config {cfg} selesai | shape={emb_mat.shape} | "
        f"total={elapsed_sec:.2f} s ({ms_per_doc:.2f} ms/dok) | "
        f"avg_tokens={doc_token_stats[cfg]['avg_tokens']} | "
        f"truncated={doc_token_stats[cfg]['pct_truncated']:.2f}%"
    )

# Simpan satu doc_ids.json yang dipakai bersama oleh A, B, dan C
with (EMB_DIR / "doc_ids.json").open("w", encoding="utf-8") as f:
    json.dump(doc_ids, f, indent=2)

encoding_log = {
    "hardware": {
        "device": DEVICE,
        "gpu_name": GPU_NAME,
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "transformers_version": transformers.__version__,
        "adapters_version": adapters.__version__,
    },
    "hyperparameters": {
        "base_model": BASE_MODEL,
        "query_adapter": QUERY_ADAPTER,
        "doc_adapter": DOC_ADAPTER,
        "sep_token": sep_token,
        "pooling": "cls_last_hidden_state",
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
    },
    "query_encoding": {
        "num_queries": len(query_ids),
        "total_time_sec": round(float(q_time_sec), 4),
        "ms_per_query": round(float((q_time_sec / len(query_ids)) * 1000.0), 4),
        **query_tok_stats,
    },
    "configs": config_logs,
}

with (EMB_DIR / "encoding_log.json").open("w", encoding="utf-8") as f:
    json.dump(encoding_log, f, indent=2)

print("\nSeluruh matriks dan encoding_log.json berhasil disimpan!")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

pytorch_adapter.bin:   0%|          | 0.00/3.59M [00:00<?, ?B/s]

Adapter dokumen aktif: Stack[proximity]


... (3 outputs left out)

Encode 5,183 Docs — Config B (proximity):   0%|          | 0/162 [00:00<?, ?it/s]

Config B selesai | shape=(5183, 768) | total=153.17 s (29.55 ms/dok) | avg_tokens=278.7 | truncated=3.90%


Encode 5,183 Docs — Config C (proximity):   0%|          | 0/162 [00:00<?, ?it/s]

Config C selesai | shape=(5183, 768) | total=160.66 s (31.00 ms/dok) | avg_tokens=297.19 | truncated=4.73%

Seluruh matriks dan encoding_log.json berhasil disimpan!


In [6]:
# 6. Uji Kewarasan (Sanity Check) Ketat & Pengemasan ZIP untuk Diunduh ke Laptop
q_check = np.load(EMB_DIR / "query_emb.npy")
da_check = np.load(EMB_DIR / "doc_emb_A.npy")
db_check = np.load(EMB_DIR / "doc_emb_B.npy")
dc_check = np.load(EMB_DIR / "doc_emb_C.npy")

assert q_check.shape == (300, 768), f"Shape query_emb salah: {q_check.shape}"
assert da_check.shape == db_check.shape == dc_check.shape == (5183, 768), "Shape doc_emb salah!"

for name, arr in [("query_emb", q_check), ("doc_emb_A", da_check), ("doc_emb_B", db_check), ("doc_emb_C", dc_check)]:
    assert arr.dtype == np.float32, f"Dtype {name} bukan float32: {arr.dtype}"
    assert not np.isnan(arr).any(), f"Ditemukan NaN pada {name}!"
    assert not np.isinf(arr).any(), f"Ditemukan Inf pada {name}!"
    norms = np.linalg.norm(arr, axis=1)
    assert np.all(norms > 0), f"Terdapat baris bernorma <= 0 pada {name}!"
    print(f"[OK] {name:10s} | shape={arr.shape} | dtype={arr.dtype} | min_norm={norms.min():.4f} | mean_norm={norms.mean():.4f}")

assert not np.allclose(da_check, db_check), "doc_emb_A dan doc_emb_B tidak boleh identik!"
assert not np.allclose(db_check, dc_check), "doc_emb_B dan doc_emb_C tidak boleh identik!"
print("\n=== SELURUH UJI KEWARASAN (SANITY CHECK) LOLOS 100%! ===")

# Kemas ke dalam 1 file ZIP agar mudah diunduh utuh ke laptop
zip_out = WORK_DIR / "embeddings_specter2.zip"
target_files = [
    "query_emb.npy",
    "query_ids.json",
    "doc_emb_A.npy",
    "doc_emb_B.npy",
    "doc_emb_C.npy",
    "doc_ids.json",
    "encoding_log.json",
]
with zipfile.ZipFile(zip_out, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in target_files:
        zf.write(EMB_DIR / fname, arcname=fname)

print(f"Arsip siap unduh: {zip_out} ({zip_out.stat().st_size / 1048576:.2f} MB)")

if IN_COLAB and USE_GOOGLE_DRIVE_BACKUP:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    backup_dir = Path("/content/drive/MyDrive/metopen_specter2_backup")
    backup_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_out, backup_dir / "embeddings_specter2.zip")
    print(f"Cadangan tersimpan di Google Drive: {backup_dir / 'embeddings_specter2.zip'}")

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_out))

[OK] query_emb  | shape=(300, 768) | dtype=float32 | min_norm=22.2529 | mean_norm=22.6696
[OK] doc_emb_A  | shape=(5183, 768) | dtype=float32 | min_norm=21.1936 | mean_norm=21.9759
[OK] doc_emb_B  | shape=(5183, 768) | dtype=float32 | min_norm=21.5214 | mean_norm=21.9809
[OK] doc_emb_C  | shape=(5183, 768) | dtype=float32 | min_norm=21.4849 | mean_norm=21.9849

=== SELURUH UJI KEWARASAN (SANITY CHECK) LOLOS 100%! ===
Arsip siap unduh: /content/embeddings_specter2.zip (43.13 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>